## This Notebook will be used to clean the raw data into processed data  
please run the cells in the given order

In [1]:
#importing necessary libraries and setting up paths. do not update this cell. Run first before running any other cell.
import sys
import shutil
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

visitors_path = (raw_path / "visitors.parquet").as_posix()
sessions_path = (raw_path / "sessions.parquet").as_posix()
outcomes_path = (raw_path / "questionnaire_outcomes.parquet").as_posix()
events_path = (raw_path / "questionnaire_events.parquet").as_posix()
answers_path = (raw_path / "questionnaire_answers.parquet").as_posix()

In [2]:
# Load raw datasets
## This cell must be executed before any cleaning or processing steps. It loads the raw datasets into pandas DataFrames for further analysis and cleaning.

visitors = pd.read_parquet(visitors_path)
sessions = pd.read_parquet(sessions_path)
events = pd.read_parquet(events_path)
answers = pd.read_parquet(answers_path)
outcomes = pd.read_parquet(outcomes_path)

In [3]:
#Using script from src/cleaning/sessions.py to detect bad visitors.
from src.cleaning.consistency import find_inconsistent_visitors

bad_ids = find_inconsistent_visitors(outcomes)

print(f"Number of inconsistent visitors: {len(bad_ids)}")

Number of inconsistent visitors: 175


In [4]:
from src.cleaning.consistency import remove_inconsistent_visitors

visitors, sessions, events_clean, answers_clean, outcomes_clean = remove_inconsistent_visitors(
    visitors, sessions, events, answers, outcomes, bad_ids
)

In [5]:
#Using script from src/cleaning/visitors.py to clean the visitors data.
from src.cleaning.visitors import normalize_visitors

visitors_clean = normalize_visitors(visitors)

In [6]:
#using script from src/cleaning/sessions.py to clean the sessions data.
from src.cleaning.sessions import normalize_sessions

sessions_clean = normalize_sessions(sessions)

In [7]:
#Saving the cleaned visitors and sessions data to the processed folder in parquet format.
visitors_clean.to_parquet(processed_path / "visitors.parquet", index=False)
sessions_clean.to_parquet(processed_path / "sessions.parquet", index=False)
events_clean.to_parquet(processed_path / "questionnaire_events.parquet", index=False)
answers_clean.to_parquet(processed_path / "questionnaire_answers.parquet", index=False)
outcomes_clean.to_parquet(processed_path / "questionnaire_outcomes.parquet", index=False)

shutil.copy2(
    raw_path / "questionnaire_questions.parquet",
    processed_path / "questionnaire_questions.parquet"
)

WindowsPath('../data/processed/questionnaire_questions.parquet')

In [8]:
#Using DuckDB to test the cleaned data.
import duckdb

processed_outcomes = (processed_path / "questionnaire_outcomes.parquet").as_posix()

duckdb.sql(f"""
WITH x AS (
    SELECT *,
        CASE outcome_category
            WHEN 'no_current_indication' THEN 1
            WHEN 'possible_risk' THEN 2
            WHEN 'declared_diagnosed' THEN 3
        END AS level,
        LAG(level) OVER (PARTITION BY visitor_id ORDER BY completed_at) AS prev
    FROM read_parquet('{processed_outcomes}')
)

SELECT COUNT(*) AS inconsistent_visitors
FROM (
    SELECT visitor_id
    FROM x
    GROUP BY visitor_id
    HAVING SUM(CASE WHEN prev = 3 AND level < 3 THEN 1 ELSE 0 END) > 0
)
""").df()

,inconsistent_visitors
0,0


In [9]:
duckdb.sql(f"""
SELECT COUNT(*) AS processed_visitors
FROM read_parquet('{(processed_path / "visitors.parquet").as_posix()}')
""").df()

,processed_visitors
0,19825


Lightweight cleaning, we identified suspected troll behavior, removed the affected visitors and their related records across all tables, and standardized the gender and region fields.

I still copied Questionnaire_questions into the processed folder for confort purposes even tho no transformation was required. For larger datasets, a different approach could be considered.